In [1]:
# Cell 1: repo / Colab setup
import os
import sys
import subprocess

REPO_DIR = "/content/OpenEnv-WolfeClick"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Atharva2099/OpenEnv-WolfeClick.git", REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Repo ready at {REPO_DIR}")


Repo ready at /content/OpenEnv-WolfeClick


In [4]:
# Cell 2: install runtime deps
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "poke-env>=0.8.0,<0.9.0",
        "transformers",
        "peft",
        "accelerate",
        "sentencepiece",
    ],
    check=True,
)

print("Dependencies installed.")


Dependencies installed.


In [5]:
# Cell 2: start local Pokemon Showdown server
import socket
import time

SHOWDOWN_DIR = "/content/pokemon-showdown"

if not os.path.exists(SHOWDOWN_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/smogon/pokemon-showdown.git", SHOWDOWN_DIR],
        check=True,
    )

subprocess.run(["pkill", "-f", "pokemon-showdown"], capture_output=True)
time.sleep(1)

showdown_proc = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    cwd=SHOWDOWN_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(20):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = s.connect_ex(("127.0.0.1", 8000))
    s.close()
    if result == 0:
        break
    time.sleep(0.5)
else:
    raise RuntimeError("Pokemon Showdown server not reachable on port 8000")

print(f"Pokemon Showdown server started (PID {showdown_proc.pid})")


Pokemon Showdown server started (PID 5030)


In [7]:
# Cell 3: load repo imports and model checkpoint
import json
import re
import webbrowser

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from smogon_rl.config import EnvConfig
from smogon_rl.openenv_sync_env import PokemonShowdownEnv
from smogon_rl.action_space import (
    enumerate_actions,
    build_action_instructions,
    extract_action_json_from_text,
    parse_llm_action,
)

HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
WATCH_REPO_ID = "Atharva2099/openenv-smogon-rl"
WATCH_REVISION = "grpo-qwen3-4b-run3"
SYSTEM_PROMPT = (
    "You are a competitive Pokemon battler. "
    "Analyse the battle state below and choose exactly one action. "
    "You MUST output ONLY a single JSON object — no explanation, no markdown fences, just the raw JSON.\n"
    '{"action": "move" | "switch", "choice": "Exact Name of Move or Pokemon"}'
)
MAX_NEW_TOKENS = 24
TEMPERATURE = 0.3
NUM_GENERATIONS = 4

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    use_fast=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(
    model,
    WATCH_REPO_ID,
    revision=WATCH_REVISION,
    token=HF_TOKEN,
    is_trainable=False,
    autocast_adapter_dtype=False,
)
model.eval()
if hasattr(model, "config"):
    model.config.use_cache = True

def build_prompt_messages(state_str, valid_actions):
    instructions = build_action_instructions(valid_actions)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{state_str}\n\n{instructions}"},
    ]

@torch.no_grad()
def generate_action_candidates(model, tokenizer, state_str, valid_actions):
    messages = build_prompt_messages(state_str, valid_actions)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": False},
    )
    device = model.get_input_embeddings().weight.device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_return_sequences=NUM_GENERATIONS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]

print(f"Loaded adapter {WATCH_REPO_ID}@{WATCH_REVISION}")


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/132M [00:00<?, ?B/s]

Loaded adapter Atharva2099/openenv-smogon-rl@grpo-qwen3-4b-run3


In [9]:
# Start local Pokemon Showdown server robustly
import os
import socket
import subprocess
import time

SHOWDOWN_DIR = "/content/pokemon-showdown"

if not os.path.exists(SHOWDOWN_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/smogon/pokemon-showdown.git", SHOWDOWN_DIR],
        check=True,
    )

subprocess.run(["pkill", "-f", "pokemon-showdown"], capture_output=True)
time.sleep(2)

showdown_proc = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    cwd=SHOWDOWN_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

server_up = False
for _ in range(40):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = s.connect_ex(("127.0.0.1", 8000))
    s.close()
    if result == 0:
        server_up = True
        break
    time.sleep(0.5)

if not server_up:
    print("Showdown failed to start. Logs:")
    try:
        print(showdown_proc.stdout.read())
    except Exception:
        pass
    raise RuntimeError("Pokemon Showdown server not reachable on port 8000")

print(f"Pokemon Showdown server started (PID {showdown_proc.pid})")


Pokemon Showdown server started (PID 7443)


In [10]:
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = s.connect_ex(("127.0.0.1", 8000))
s.close()
print("server up" if result == 0 else "server down")


server up


In [11]:
# Cell 4: run one live battle and print the room URL
OPEN_BROWSER = True

env = PokemonShowdownEnv(
    config=EnvConfig(
        battle_format="gen4randombattle",
        verbose_logging=True,
        log_every_n_steps=1,
        poll_heartbeat_seconds=5.0,
        max_steps_per_battle=30,
    )
)

state_str = env.reset()
battle = env._ensure_battle()
room_path = f"/{battle.battle_tag}"
room_url = f"http://127.0.0.1:8000{room_path}"
print("Battle room path:", room_path)
print("Local Showdown URL:", room_url)
print("If this kernel is remote, use VS Code port forwarding for port 8000.")
if OPEN_BROWSER:
    webbrowser.open(room_url)

done = False
total_reward = 0.0
step_idx = 0

while not done:
    step_idx += 1
    battle = env._ensure_battle()
    valid_actions = enumerate_actions(battle)
    if not valid_actions:
        print("No valid actions available; ending battle.")
        break

    candidates = generate_action_candidates(model, tokenizer, state_str, valid_actions)
    chosen_str = None

    for c in candidates:
        clean = re.sub(r"<think>.*?</think>", "", c, flags=re.DOTALL).strip()
        try:
            parse_llm_action(clean, valid_actions)
            chosen_str = clean
            break
        except ValueError:
            pass

        extracted = extract_action_json_from_text(c)
        if extracted is not None:
            try:
                parse_llm_action(extracted, valid_actions)
                chosen_str = extracted
                break
            except ValueError:
                pass

    if chosen_str is None:
        fb = valid_actions[0]
        chosen_str = json.dumps({"action": fb.action_type, "choice": fb.choice})

    state_str, reward, done, info = env.step(chosen_str)
    total_reward += float(reward)
    print(
        f"step={step_idx} turn={info.get('turn')} action={chosen_str} "
        f"reward={reward:.3f} total_reward={total_reward:.3f} done={done}",
        flush=True,
    )

finished_battle = env._ensure_battle()
print(
    f"Battle complete | won={getattr(finished_battle, 'won', None)} "
    f"lost={getattr(finished_battle, 'lost', None)} total_reward={total_reward:.3f}"
)
print("Room path:", room_path)
print("Local URL:", room_url)


[PokeEnvClient] Background event loop started.
[PokeEnvClient] Players created and connected.
[PokeEnvClient] Launching new battle in format gen4randombattle.
[PokeEnvClient] Battle update received: turn=1, finished=False.
[PokemonShowdownEnv] Battle 1 started at turn=1 (format=gen4randombattle).
Battle room path: /battle-gen4randombattle-1
Local Showdown URL: http://127.0.0.1:8000/battle-gen4randombattle-1
If this kernel is remote, use VS Code port forwarding for port 8000.
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=10.0s.
[PokeEnvClient] Battle update received: turn=2, finished=False.
[PokemonShowdownEnv] battle=1 step=1 turn=2 reward=2.489 running_reward=2.489 illegal_action=False done=False
step=1 turn=2 action={"action": "move", "choice": "poisonjab"} reward=2.489 total_reward=2.489 done=False
[PokeEnvC

In [12]:
battle = env._ensure_battle()

out = {
    "room_path": room_path,
    "won": getattr(battle, "won", None),
    "lost": getattr(battle, "lost", None),
    "turn": getattr(battle, "turn", None),
    "total_reward": total_reward,
    "battle_tag": getattr(battle, "battle_tag", None),
}

print(json.dumps(out, indent=2))


{
  "room_path": "/battle-gen4randombattle-1",
  "won": null,
  "lost": null,
  "turn": 31,
  "total_reward": 7.615827366746224,
  "battle_tag": "battle-gen4randombattle-1"
}
